In [1]:
import textgrad as tg
from textgrad.tasks import Dataset
import pandas as pd
import ast
import random
from typing import List, Callable
from instructions import *
from few_shot_examples import *
from utils import *
from dotenv import load_dotenv
from openai import OpenAI
from operating_points import get_sub_questions
from sync_eval import AutoEvaluator
load_dotenv(override=True)
OPENAI_API_KEY = os.environ['OPENAI_API_KEY'] 

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/theodoraworledge/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
# Set up a client for non-substantive LLM calls for formatting
helper_client = OpenAI(api_key=OPENAI_API_KEY)

# Set-up the prompts
rubric_prompt = tg.Variable(
    hand_written_outer_prompt_generation_str, # "Ground your response to the query in the sources provided.", # TODO use hand_written_outer_prompt_generation_str?
    role_description='A system prompt that specifies how to answer a query using the sources provided.',
    requires_grad=True,
)

cite_rubric_prompt = tg.Variable(
    hand_written_outer_prompt_citation_id_str, # "Identify the quotes that support the sentence.", # TODO use a variant of hand_written_outer_prompt_generation_str?
    role_description='A system prompt that specifies how to identify the quotes a sentence should cite.',
    requires_grad=True,
)

quoted_prompt = tg.Variable(
    construct_prompt(gold_quote_cot_instruction_str, quote_cot_few_shot_examples_dict, just_prompt_and_fewshot=True),
    role_description='A prompt that specifies how to answer a query using quotes from sources.',
    requires_grad=False,
)

paraphrased_prompt = tg.Variable(
    construct_prompt(paraphrase_instruction_str, paraphrase_few_shot_examples_dict, just_prompt_and_fewshot=True),
    role_description='A prompt that specifies how to answer a query using an entailed revision of a response.',
    requires_grad=False,
)

entailed_prompt = tg.Variable(
    construct_prompt(entailment_instruction_str, entailment_few_shot_examples_dict, just_prompt_and_fewshot=True),
    role_description='A prompt that specifies how to answer a query using an entailed revision of a response.',
    requires_grad=False,
)

abstractive_prompt = tg.Variable(
    construct_prompt(abstractive_instruction_str, abstractive_few_shot_examples_dict, just_prompt_and_fewshot=True),
    role_description='A prompt that specifies how to answer a query using an abstractive revision of a response.',
    requires_grad=False,
)

forward_engine = tg.get_engine("experimental:gpt-4", cache=False)
generate_fn = tg.autograd.LLMCall(engine=forward_engine, system_prompt=rubric_prompt)

forward_engine = tg.get_engine("experimental:gpt-4", cache=False)
cite_fn = tg.autograd.LLMCall(engine=forward_engine, system_prompt=cite_rubric_prompt)

backward_engine = tg.get_engine("experimental:gpt-4.1", cache=False)
tg.set_backward_engine(backward_engine, override=True)

loss_system_prompt = tg.Variable(f"""You will evaluate a response to a query, given the goals of following the inner instructions and jointly improving citation precision, citation recall, perceived utility, and fluency. Be super concise. Only identify errors and areas for improvement.

Fluency: {autoeval_fluency_instruction}

Perceived utility: {autoeval_perceived_utility_instruction}

Citation precision: You are an intelligent and fair attribution evaluator. Your task is to verify whether each citation or quote supports at least one claim made in the sentence that contains it. Citations and quotes should not be irrelevant or contradictory to the sentence that contains them.

Citation coverage: You are an intelligent and fair attribution evaluator. Your task is to verify whether the citation(s) or quote(s) cited in each sentence support all of the claims made in that sentence.

Prioritization: Your task is to ensure that the response is prioritizing its instructions to quote, paraphrase, and entail above all else.
""",
    requires_grad=True,
    role_description="Feedback on a response to a query for fluency, perceived utility, citation precision, and citation coverage.")

loss_fn = tg.TextLoss(loss_system_prompt)
optimizer = tg.TGD([rubric_prompt, cite_rubric_prompt])

In [3]:
class QuerySourceDataset(Dataset):
    def __init__(self, num_samples, seed):
        np.random.seed(seed)
        random.seed(seed)
        self.nq_df = pd.read_csv('autoEval_results/autoEval_gpt4_nq_byQueryOP.csv')
        self.nq_df['data_str'] = 'nq'
        self.eli5_df = pd.read_csv('autoEval_results/autoEval_gpt4_eli5_nq_byQueryOP.csv')
        self.eli5_df['data_str'] = 'eli5_nq'
        self.multihop_df = pd.read_csv('autoEval_results/autoEval_gpt4_multihop_byQueryOP.csv')
        self.multihop_df['data_str'] = 'multihop'
        self.mash_df = pd.read_csv('autoEval_results/autoEval_gpt4_mash_byQueryOP.csv')
        self.mash_df['data_str'] = 'mash'

        def get_source_str(value):
            value = ast.literal_eval(value)
            source_str = ""
            for i, source in enumerate(value):
                source_str += f"\"{source}\"\n\n"
            return source_str

        def get_numbered_source_str(value):
            source_ls = ast.literal_eval(value)
            source_str = "\n\nSources:\n\n"
            for source in source_ls:
                source_str += "\""
                source = source.strip()
                source_sentence_ls = source.split(".")
                for i, sentence in enumerate(source_sentence_ls):
                    
                    source_str += f" [{i+1}] {sentence}."
                source_str += "\"\n\n"
            return source_str

        num_samples_per_op = num_samples // 4 // 3
        # For the quoted OP, add a citation marker for each sentence in the sources.
        quoted_sample = self.nq_df.copy(deep=True).iloc[0:0]
        for df in [self.nq_df, self.eli5_df, self.multihop_df, self.mash_df]:
            quoted_sample = pd.concat([quoted_sample, df[df['op']=='Quoted'].sample(num_samples_per_op)])

        # Add a citation marker before each sentence in the sources.
        quoted_sample['Sources String'] = quoted_sample['All Sources'].apply(get_numbered_source_str)
        # quoted_sample = quoted_sample[['Sources String', 'Question', 'op', 'data_str', 'A']]

        # For the paraphrased and entailed OPs, use the Used Sources (cited) String column
            # Also, provide the quoted response
        paraphrased_and_entailed_sample = self.nq_df.copy(deep=True).iloc[0:0]
        for df in [self.nq_df, self.eli5_df, self.multihop_df, self.mash_df]:
            paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Paraphrased'].sample(num_samples_per_op)])

        for df in [self.nq_df, self.eli5_df, self.multihop_df, self.mash_df]:
            paraphrased_and_entailed_sample = pd.concat([paraphrased_and_entailed_sample, df[df['op']=='Entailed'].sample(num_samples_per_op)])

        paraphrased_and_entailed_sample['Sources String'] = paraphrased_and_entailed_sample['Used Sources (cited)'].apply(get_source_str)

        # Add the quoted response for each query to the front of the Source String
        all_df = pd.concat([self.nq_df, self.eli5_df, self.multihop_df, self.mash_df])
        quoted_responses = []
        questions = []
        for question in paraphrased_and_entailed_sample['Question']:
            quoted_response = all_df[(all_df['Question'] == question) & (all_df['op'] == 'Quoted')]['Output (cited)'].iloc[0]
            quoted_responses.append(quoted_response)
            questions.append(question)

        quoted_response_df = pd.DataFrame({'Quoted Response': quoted_responses, 'Question': questions})

        paraphrased_and_entailed_sample = pd.merge(paraphrased_and_entailed_sample, quoted_response_df, on='Question', how='left')
        # paraphrased_and_entailed_sample = paraphrased_and_entailed_sample[['Sources String', 'Question', 'op', 'data_str']]

        training_data = pd.concat([quoted_sample, paraphrased_and_entailed_sample])
        training_data = training_data.sample(len(training_data)).reset_index(drop=True)
        self.training_data = training_data

    def __getitem__(self, index):
        query = self.training_data.iloc[index]['Question']
        source_str = self.training_data.iloc[index]['Sources String']
        data_str = self.training_data.iloc[index]['data_str']
        urls = ast.literal_eval(self.training_data.iloc[index]['All URLs'])
        sources = ast.literal_eval(self.training_data.iloc[index]['All Sources'])
        sources_with_urls = [u+'\n'+s for u, s in zip(urls, sources)]

        return query, source_str, data_str, sources, sources_with_urls

    def __len__(self):
        return len(self.training_data)


In [4]:
class CiteQuotedResponse(tg.autograd.function.Module):
    def __init__(self, sources, client, data_str):
        self.sources = sources
        self.client = client
        self.data_str = data_str

    def forward(self, uncited_quoted_response):
        self.uncited_quoted_response = uncited_quoted_response
        highlighted_quoted_response, highlighted_sources, sources_idxs, used_sources, used_unmarked_sources, used_highlighted_cited_sources, m_sentences_quoted_response, sentences_quoted_response, quoted_num_cited, quoted_citations_dict, used_highlighted_uncited_sources = highlight_direct_quotes(uncited_quoted_response.value, self.sources, self.client, self.data_str)
        highlighted_quoted_response_var = tg.Variable(highlighted_quoted_response, 
                                                      role_description='A quoted response with citations.', 
                                                      requires_grad=True,
                                                      predecessors=[uncited_quoted_response])
        return highlighted_quoted_response_var, highlighted_sources, sources_idxs, used_sources, used_unmarked_sources, used_highlighted_cited_sources, m_sentences_quoted_response, sentences_quoted_response, quoted_num_cited, quoted_citations_dict, used_highlighted_uncited_sources 
    
    def backward(self, grad_output):
        # Pass along the gradients
        return grad_output

class CitedResponseGenerator(tg.autograd.function.Module):
    def __init__(self, client, data_str, op, uncited_quoted_response, sources_idxs):
        self.client = client
        self.data_str = data_str
        self.op = op
        self.sources_idxs = sources_idxs
        # identify all quoted sentences
        self.uncited_quoted_response = uncited_quoted_response
        self.quoted_sentences_ls = get_quoted_sentences(uncited_quoted_response.value, self.client, data_str=self.data_str)
        self.quotes_str = '\nQuotes:\n'+format_quoted_sentences_ls(self.quoted_sentences_ls)

    def forward(self, uncited_op_response):
        # Store for the backward pass
        self.uncited_op_response = uncited_op_response

        # Identify all paraphrased sentences
        op_sentences = get_sentences_gpt4(uncited_op_response.value, self.client)
        self.op_sentences = op_sentences

        cited_op_sentence_vars = []
        cited_op_sentences = []
        citations_dict = {}
        num_citations = 0
        for i in range(len(op_sentences)):
            citations_dict[i] = {'citation_numbers': []}
            op_sentence = op_sentences[i]
            if (len(op_sentence) == 0):
                continue

            op_sentence_var = tg.Variable(op_sentence, 
                                          role_description='A sentence from a paraphrased or entailed response.', 
                                          requires_grad=True,
                                          predecessors=[uncited_op_response])

            text_str_var = tg.Variable('Text: ', 
                                        role_description='A label for the text string.', 
                                        requires_grad=False)

            few_shot_examples_dict = get_few_shot_examples_for_cite_pp(self.op, self.data_str)
            prompt = construct_prompt(id_pp_ent_abs_citations_instruction_str, few_shot_examples_dict, [None, None], '', just_prompt_and_fewshot=True)
            prompt_var = tg.Variable(prompt, 
                                    role_description='A prompt that specifies how to identify the quotes a sentence should cite and provides few shot examples.', 
                                    requires_grad=False)
            quotes_str_var = tg.Variable(self.quotes_str, 
                                        role_description='A list of quotes, which are each candidate citations.', 
                                        requires_grad=True)
            citation_response_str_var = tg.Variable("\nAnswer: ", 
                                                    role_description='A response to a citation request.', 
                                                    requires_grad=False)
            # Cite this sentence
            citations_var = cite_fn(prompt_var+text_str_var+op_sentence_var+quotes_str_var+citation_response_str_var) 

            # Parse the citation numbers
            proposed_citation_numbers = re.findall(r'\[(\d+)\]', citations_var.value)
            citation_numbers = ' '
            citation_numbers_ls = []
            for proposed_citation_number in proposed_citation_numbers:
                citation_number = int(proposed_citation_number)
                if (citation_number-1 >= len(self.sources_idxs)):
                    return None, None, None, None, None
                if (self.sources_idxs[citation_number-1] != None): # only keep citation if the quote is precise
                    source_idx = self.sources_idxs[citation_number-1]['source_idx']
                    citation_numbers += COLORS[source_idx]+"["+str(citation_number)+"]"+COLORS[10]
                    citation_numbers_ls.append(citation_number)
                    num_citations += 1

            citations_dict[i]['citation_numbers'].extend(citation_numbers_ls)
            if (citation_numbers == ' '):
                citation_numbers = ''
                
            # remove the end punctuation to keep the reference inside the sentence it refers to
            end_punctuation = ''
            if ((op_sentence[-1]=='.') or (op_sentence[-1]==',') or (op_sentence[-1]=='!') or (op_sentence[-1]=='?')):
                end_punctuation = op_sentence[-1]
                op_sentence = op_sentence[:-1]

            # paraphrased_response = paraphrased_response.replace(paraphrased_sentence, paraphrased_sentence+citation_numbers)
            op_sentence = op_sentence+citation_numbers+end_punctuation
            cited_op_sentences.append(op_sentence)
            cited_op_sentence_var = tg.Variable(op_sentence, 
                                          role_description='A cited sentence from a paraphrased or entailed response.', 
                                          requires_grad=True,
                                          predecessors=[op_sentence_var])
            cited_op_sentence_vars.append(cited_op_sentence_var)

        # Recombine the cited sentences
        cited_response_var = tg.autograd.functional.aggregate(cited_op_sentence_vars)
        for i in range(len(cited_op_sentences)):
            cited_op_sentences[i] = cited_op_sentences[i].replace("'", "\'")
            op_sentences[i] = op_sentences[i].replace("'", "\'")

        return cited_response_var, cited_op_sentences, op_sentences, citations_dict

def generate_all_ops(query, sub_queries, sources, sources_str, data_str, for_eval=False):

    # Generate quoted response
    quoted_input = tg.Variable(f"""
Query: {query}
Sub-questions: {sub_queries}
Sources:
{sources_str}
""",
        role_description='A query to answer given sub-questions and sources.',
        requires_grad=False,
    )
    quoted_response = generate_fn(quoted_prompt + quoted_input)
    g_cited_quoted_response = CiteQuotedResponse(sources, helper_client, data_str)
    cited_quoted_response_var, highlighted_sources, sources_idxs, used_sources, used_unmarked_sources, used_highlighted_cited_sources, m_sentences_quoted_response, sentences_quoted_response, quoted_num_cited, quoted_citations_dict, used_highlighted_uncited_sources = g_cited_quoted_response(quoted_response)

    # Generate paraphrased response
    paraphrased_input = tg.Variable(f"""
Query: {query}
Response: """,
        role_description='A paraphrased response to a query.',
        requires_grad=False,
    )
    paraphrased_input_end = tg.Variable(
        "\nParaphrased Response: ",
        role_description='A paraphrased response to a query.',
        requires_grad=False,
    )

    paraphrased_response = generate_fn(paraphrased_prompt + paraphrased_input + quoted_response + paraphrased_input_end)
    g_cited_paraphrased_response = CitedResponseGenerator(helper_client, data_str, 'paraphrased', quoted_response, sources_idxs)
    cited_paraphrased_response_var, cited_paraphrased_sentences, paraphrased_sentences, paraphrased_citations_dict = g_cited_paraphrased_response(paraphrased_response)

    # Generate entailed response
    entailed_input = tg.Variable(f"""
Query: {query}
Response: """,
        role_description='An entailed response to a query.',
        requires_grad=False,
    )
    entailed_input_end = tg.Variable(
        "\nEntailed Response: ",
        role_description='An entailed response to a query.',
        requires_grad=False,
    )

    entailed_response = generate_fn(entailed_prompt + entailed_input + quoted_response + entailed_input_end)

    g_cited_entailed_response = CitedResponseGenerator(helper_client, data_str, 'entailed', quoted_response, sources_idxs)
    cited_entailed_response_var, cited_entailed_sentences, entailed_sentences, entailed_citations_dict = g_cited_entailed_response(entailed_response)


    # citation_dict, sent_ls, cited_source_ls, source_ls
    if not for_eval:
        return cited_quoted_response_var, cited_paraphrased_response_var, cited_entailed_response_var
    else:
        return cited_quoted_response_var, cited_paraphrased_response_var, cited_entailed_response_var, \
        quoted_citations_dict, paraphrased_citations_dict, entailed_citations_dict, \
        sentences_quoted_response, paraphrased_sentences, entailed_sentences, \
        used_highlighted_cited_sources, \
        quoted_response, paraphrased_response, entailed_response
        
        


In [5]:
validation_data = QuerySourceDataset(36, 0) 
validation_loader = tg.tasks.DataLoader(validation_data, batch_size=1, shuffle=True)

def validate_new_system_prompts():
    fluency_scores_ls = []
    perceived_utility_scores_ls = []
    precision_scores_ls = []
    coverage_scores_ls = []
    
    for batch in validation_loader:
        batch_query, batch_sources_str, batch_data_str, batch_sources, batch_sources_with_urls = batch
        query = batch_query[0]
        sources_str = batch_sources_str[0]
        data_str = batch_data_str[0]
        sources = batch_sources[0]
        sources_with_urls = batch_sources_with_urls[0]
        sub_queries = get_sub_questions(query, helper_client)

        cited_quoted_response_var, cited_paraphrased_response_var, cited_entailed_response_var, \
        quoted_citations_dict, paraphrased_citations_dict, entailed_citations_dict, \
        sentences_quoted_response, paraphrased_sentences, entailed_sentences, \
        used_highlighted_cited_sources, \
        quoted_response, paraphrased_response, entailed_response = generate_all_ops(query, sub_queries, sources, sources_str, data_str, for_eval=True)

        responses = [quoted_response, paraphrased_response, entailed_response]
        op_strs = ['quoted', 'paraphrased', 'entailed']
        citations_dicts = [quoted_citations_dict, paraphrased_citations_dict, entailed_citations_dict]
        sent_ls_ls = [sentences_quoted_response, paraphrased_sentences, entailed_sentences]
        cited_source_ls_ls = [used_highlighted_cited_sources, used_highlighted_cited_sources, used_highlighted_cited_sources]
        source_ls_ls = [sources, sources, sources]
        
        evaluator = AutoEvaluator()

        for response, op_str, citation_dict, sent_ls, cited_source_ls, source_ls in zip(responses, op_strs, citations_dicts, sent_ls_ls, cited_source_ls_ls, source_ls_ls):
            fluency_scores = evaluator.evaluate_fluency(query, response.value)
            perceived_utility_scores = evaluator.evaluate_perceived_utility(query, response.value)
            precision_scores, t2v_precision = evaluator.evaluate_response_citation_precision(citation_dict, sent_ls, cited_source_ls, source_ls)
            coverage_scores, t2v_coverage = evaluator.evaluate_response_citation_coverage(citation_dict, sent_ls, cited_source_ls, source_ls)
            fluency_scores_ls.append(fluency_scores)
            perceived_utility_scores_ls.append(perceived_utility_scores)
            precision_scores_ls.extend(precision_scores)
            coverage_scores_ls.extend(coverage_scores)
    
    avg_fluency_score = np.mean(fluency_scores_ls)
    avg_utility_score = np.mean(perceived_utility_scores_ls)
    avg_coverage_score = 0
    num_sentences = 0
    for coverage_scores in coverage_scores_ls:
        num_sentences += 1
        avg_coverage_score += coverage_scores['coverage']
    avg_coverage_score /= num_sentences

    avg_precision_score = 0
    num_citations = 0
    for precision_scores in precision_scores_ls:
        num_citations += len(precision_scores['annotations'])
        if (len(precision_scores['annotations']) > 0):
            avg_precision_score += sum(precision_scores['annotations'])
    avg_precision_score /= num_citations

    print("avg_fluency_score", avg_fluency_score)
    print("avg_utility_score", avg_utility_score)
    print("avg_coverage_score", avg_coverage_score)
    print("avg_precision_score", avg_precision_score)
            
    return avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score


In [6]:
def aggregate_results(avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score):
    # return min(avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score)
    return (avg_fluency_score-1)/2 + (avg_utility_score-1)/2 + avg_coverage_score + avg_precision_score

In [7]:
results = {}
# Create the training data
batch_size = 3
training_data = QuerySourceDataset(36, 21)
train_loader = tg.tasks.DataLoader(training_data, batch_size=batch_size, shuffle=True)

avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score = validate_new_system_prompts()
previous_overall_score = aggregate_results(avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score)
previous_rubric_prompt = rubric_prompt.value
previous_cite_rubric_prompt = cite_rubric_prompt.value
results['rubric_prompt'] = rubric_prompt.value
results['cite_rubric_prompt'] = cite_rubric_prompt.value
results['fluency_score'] = avg_fluency_score
results['utility_score'] = avg_utility_score
results['coverage_score'] = avg_coverage_score
results['precision_score'] = avg_precision_score
results['overall_score'] = previous_overall_score

batch_idx = 0
for batch in train_loader:
    optimizer.zero_grad()
    losses = []
    batch_query, batch_sources_str, batch_data_str, batch_sources, batch_sources_with_urls = batch
    for query, sources_str, data_str, sources, sources_with_urls in zip(batch_query, batch_sources_str, batch_data_str, batch_sources, batch_sources_with_urls):
        sub_queries = get_sub_questions(query, helper_client)
        cited_quoted_response_var, cited_paraphrased_response_var, cited_entailed_response_var = generate_all_ops(query, sub_queries, sources, sources_str, data_str)
        quoted_loss = loss_fn(cited_quoted_response_var)
        paraphrased_loss = loss_fn(cited_paraphrased_response_var)
        entailed_loss = loss_fn(cited_entailed_response_var)
        loss = quoted_loss + paraphrased_loss + entailed_loss
        losses.append(loss)
    total_loss = tg.sum(losses)
    total_loss.backward()
    optimizer.step()
    print('~~~~~~~~~~~~~~~~~'*10)
    print("Optimized Rubric Prompt: \n", rubric_prompt.value)
    print('-----------------'*10)
    print("Optimized Cite Rubric Prompt: \n", cite_rubric_prompt.value)
    print('-----------------'*10)
    avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score = validate_new_system_prompts()
    new_overall_score = aggregate_results(avg_fluency_score, avg_utility_score, avg_coverage_score, avg_precision_score)
    if (new_overall_score <= previous_overall_score):
        # revert to the previous rubric and cite rubric prompts
        rubric_prompt.set_value(previous_rubric_prompt)
        cite_rubric_prompt.set_value(previous_cite_rubric_prompt)
    else:
        previous_overall_score = new_overall_score
        previous_rubric_prompt = rubric_prompt.value
        previous_cite_rubric_prompt = cite_rubric_prompt.value
        results['rubric_prompt'] = rubric_prompt.value
        results['cite_rubric_prompt'] = cite_rubric_prompt.value
        results['fluency_score'] = avg_fluency_score
        results['utility_score'] = avg_utility_score
        results['coverage_score'] = avg_coverage_score
        results['precision_score'] = avg_precision_score
        results['overall_score'] = new_overall_score
        results['batch_idx'] = batch_idx

    batch_idx += 1


avg_fluency_score 2.6052631578947367
avg_utility_score 2.5701754385964914
avg_coverage_score 0.24079320113314448
avg_precision_score 0.865814696485623
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
Optimized Rubric Prompt: 
 You are an attentive assistant who strictly follows all instructions and maximizes the following metrics: fluency, perceived utility, citation coverage, and citation precision. 

**General Principles:**
- All responses must be constructed *exclusively* from the information present in the provided sources or prior responses, unless explicitly instructed to include outside knowledge. Never hallucinate or invent information.
- Every factual or actionable statement must be traceable and fully supported by the sources. Do not omit or add facts; maintain strict one-to-one correspondence between supported facts and your output, especially when paraph

In [8]:
import json

with open('textgrad_results_2_17.json', 'w') as f:
    json.dump(results, f)

In [21]:
with open('textgrad_results_2_17.json', 'r') as f:
    results_cp = json.load(f)

# print(list(results_cp.keys()))
# print()
results_cp['rubric_prompt']

'You are an attentive assistant who strictly follows all user instructions and maximizes fluency, perceived utility, citation coverage, and citation precision. Use only information found in the provided sources (or prior responses, if explicitly allowed); never hallucinate or invent information, and do not refer to prior responses or input meta-information unless instructed.\n\n**General Principles:**\n- Build every response *exclusively* from claims directly traceable to the provided sources. Do not use outside knowledge unless explicitly allowed.\n- **For every output, regardless of length:**  \n  - **NEVER submit a blank, fragmentary, or verbatim output.** Even ultra-short, single-fact, or minimal responses require at least one overt paraphrase, format change, or direct answer structure (e.g., turning “1995” or “‘Song’ (2003)” into “The song was released in 2003.”). Never produce a blank or unchanged output—retry or rephrase until this minimum is achieved.\n  - If a maximally fluent

# Old code

In [ ]:
# Load in the data
data_str = 'nq'
query = "Who won the 2018 Olympic Women's Hockey Gold Medal?"
sub_queries = "Who won the 2018 Olympic Women's Hockey Gold Medal?"
urls = ["www.sports.com", "www.espn.com"]
sources = ["The U.S. Olympic Women’s Ice Hockey Team is 4-0 and capped preliminary play with a 5-0 win over Canada. The team faces Italy in the quarterfinals Friday at 3:10 p.m. ET (USA Network | Peacock). The United States has already started to establish its dominance last year, taking home gold medals at both senior men’s and women’s IIHF World Championships and at the World Para Ice Hockey Championship. Retrieved 2026-02-12.",
"The women's tournament in ice hockey at the 2018 Winter Olympics was held in Gangneung, South Korea between 10 and 22 February 2018.[1] Eight countries qualified for the tournament; five of them did so automatically by virtue of their ranking by the International Ice Hockey Federation, one, South Korea, automatically qualified as hosts, while the two others took part in a qualification tournament.[2] Under a special agreement with the IOC and the IIHF, twelve North Korean players joined the host team to form a united team.[3] They were allowed to have an expanded roster of 35 where 22 players dress for each game. Three North Korean players were selected for each game by coach Sarah Murray.[4] The United States winning the gold medal game against Canada marks the first time in 20 years that the United States took home a gold medal in women's hockey. They previously won in 1998 in Nagano, Japan, which was also against Canada.[5] Canada's loss ended their winning streak of four consecutive winter games, having won since 2002.[6]"]
sources_with_urls = [u+'\n'+s for u, s in zip(urls, sources)]
sources_str = "Sources:\n"+format_source_list(sources)

question_str = "Query: "+query

In [5]:

# Final cited quoted response and ls of cited sources: highlighted_quoted_response.data, used_highlighted_cited_sources
cited_quoted_response_var, cited_paraphrased_response_var, cited_entailed_response_var = generate_all_ops(query, sub_queries, sources, sources_str, data_str)

In [6]:
print(cited_quoted_response_var.value)

"The United States winning the gold medal game against Canada marks the first time in 20 years that the United States took home a gold medal in women's hockey" [1].


In [7]:
print(cited_paraphrased_response_var.value)

The victory of the United States over Canada in the gold medal match signifies the first time in two decades that the United States has clinched a gold medal in women's hockey [1].


In [8]:
print(cited_entailed_response_var.value)

The United States won the 2018 Olympic Women's Hockey Gold Medal [1].


In [ ]:
quoted_loss = loss_fn(cited_quoted_response_var)
paraphrased_loss = loss_fn(cited_paraphrased_response_var)
entailed_loss = loss_fn(cited_entailed_response_var)
loss = quoted_loss + paraphrased_loss + entailed_loss

loss.backward()
optimizer.step()


In [39]:
print(rubric_prompt.value)

Respond to the query using only information explicitly supported by the provided sources. Do not include any details, inferences, or world knowledge that are not directly stated or logically entailed by the sources. Address all parts of the query and any sub-questions, ensuring each relevant material fact or perspective from the sources appears in the answer. If the sources present conflicting or ambiguous information on a material point, represent each perspective in a separate, standalone sentence using the exact or most complete phrasing from the sources, and do not merge or summarize viewpoints. If no relevant information exists in the sources for part or all of the query, state explicitly that the answer is not available from the provided sources. 

Express each material fact as its own standalone sentence, without relying on pronouns or context from previous sentences; avoid merging or splitting facts in ways that obscure distinctions present in the originals. Maintain all import

In [40]:
print(cite_rubric_prompt.value)

Carefully examine the sentence to identify which quotes must be cited for accurate attribution. Select quotes that explicitly support, illustrate, or provide direct evidence for the claims or information presented in the sentence. Ensure that these quotes precisely correspond to both the content and the intended meaning of the sentence, avoiding unrelated or tangentially connected material.


In [43]:
graph = cited_entailed_response_var.generate_graph(print_gradients=False)
print(graph.source)

// Computation Graph starting from a combination of the following variables: A cited sentence from a paraphrased or entailed response..
digraph {
	rankdir=TB
	ranksep=0.2
	bgcolor=lightgrey
	fontsize=7.5
	13184823280 [label=<<b><font color='darkblue'>Role: </font></b> A combination of the following<br/>variables: a cited sentence from a<br/>paraphrased or entailed response..<br/><b><font color='darkblue'>Value: </font></b> The United States won the 2018 Olympic<br/>Women's Hockey Gold Medal, marking their<br/>first gold in women's hockey in 20 years<br/>[1].<br/><b><font color='darkblue'>Grad Fn: </font></b> <br/>textgrad.autograd.algebra.Aggregate.backward<br/><b><font color='darkblue'>Reduce Meta: </font></b> [{'op': &lt;function<br/>_reduce_gradients_mean at<br/>0x10445b420&gt;, 'id': 13184276416}]> fillcolor=lavender fontname=Arial fontsize=8 margin=0.1 pad=0.1 shape=rectangle style=filled width=1.2]
	13184826112 -> 13184823280
	13184826112 [label=<<b><font color='darkblue'>Role: <